# NeuralAlpha LSTM Time-Series Prediction

Lightweight LSTM pipeline for predicting future values from a time-series `df` containing a `close` column.

In [ ]:
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler

try:
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense
except Exception as exc:  # noqa: BLE001
    raise ImportError("TensorFlow is required for this notebook. Install tensorflow before running.") from exc

pd.set_option("display.max_rows", 20)
pd.set_option("display.width", 140)
np.set_printoptions(suppress=True, precision=4)

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("neuralalpha_lstm")

WINDOW_SIZE = 10
MAX_ROWS = 1000
EPOCHS = 5
BATCH_SIZE = 16
ARTIFACT_DIR = Path("ml_pipeline/models")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def get_input_dataframe() -> pd.DataFrame:
    """Return the preloaded df or load a fallback raw price dataset."""
    if "df" in globals() and isinstance(globals()["df"], pd.DataFrame):
        logger.info("Using preloaded df from notebook environment.")
        return globals()["df"].copy()

    search_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    raw_dir = None
    for root in search_roots:
        candidate = root / "ml_pipeline" / "data" / "raw"
        if candidate.exists():
            raw_dir = candidate
            break

    if raw_dir is None:
        raise FileNotFoundError("No preloaded df found and no ml_pipeline/data/raw directory was available.")

    preferred_files = [raw_dir / "stock_market_data.csv", raw_dir / "crypto_market_data.csv"]
    for file_path in preferred_files:
        if file_path.exists():
            logger.info("Loading fallback dataset: %s", file_path.name)
            return pd.read_csv(file_path)

    raise FileNotFoundError("No usable fallback dataset was found in ml_pipeline/data/raw.")


def prepare_close_series(frame: pd.DataFrame, max_rows: int = MAX_ROWS) -> np.ndarray:
    """Extract and normalize the close series using only the most recent rows."""
    data = frame.copy()
    data.columns = data.columns.astype(str).str.strip().str.lower()

    if "close" not in data.columns:
        raise ValueError("Input dataframe must contain a 'close' column.")

    if len(data) > max_rows:
        data = data.tail(max_rows).copy()
        logger.info("Using most recent %d rows for fast LSTM training.", max_rows)

    close_series = pd.to_numeric(data["close"], errors="coerce").dropna().to_numpy(dtype=np.float32)
    if close_series.size <= WINDOW_SIZE:
        raise ValueError("Not enough close values to build sequences.")

    return close_series.reshape(-1, 1)


def create_sequences(values: np.ndarray, window_size: int = WINDOW_SIZE) -> tuple[np.ndarray, np.ndarray]:
    """Create sliding window sequences efficiently with NumPy stride-like slicing."""
    flat = values.reshape(-1)
    total_sequences = len(flat) - window_size
    if total_sequences <= 0:
        raise ValueError("Not enough rows to create sequences.")

    x = np.empty((total_sequences, window_size, 1), dtype=np.float32)
    y = np.empty((total_sequences,), dtype=np.float32)

    for index in range(total_sequences):
        x[index, :, 0] = flat[index : index + window_size]
        y[index] = flat[index + window_size]

    return x, y


def build_model(window_size: int = WINDOW_SIZE) -> Sequential:
    """Build a lightweight LSTM model for fast training."""
    model = Sequential([
        LSTM(32, input_shape=(window_size, 1)),
        Dense(1),
    ])
    model.compile(optimizer="adam", loss="mse")
    return model


def train_lstm_pipeline(frame: pd.DataFrame) -> dict[str, object]:
    """Train, evaluate, and persist the LSTM pipeline artifacts."""
    close_values = prepare_close_series(frame, max_rows=MAX_ROWS)

    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_values = scaler.fit_transform(close_values)

    x, y = create_sequences(scaled_values, window_size=WINDOW_SIZE)
    split_index = int(len(x) * 0.8)
    x_train, x_test = x[:split_index], x[split_index:]
    y_train, y_test = y[:split_index], y[split_index:]

    model = build_model(window_size=WINDOW_SIZE)
    history = model.fit(
        x_train,
        y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=0,
        shuffle=False,
    )

    predictions = model.predict(x_test, verbose=0).reshape(-1, 1)
    y_test_2d = y_test.reshape(-1, 1)

    predictions_inverse = scaler.inverse_transform(predictions).ravel()
    actual_inverse = scaler.inverse_transform(y_test_2d).ravel()

    mae = mean_absolute_error(actual_inverse, predictions_inverse)
    mse = mean_squared_error(actual_inverse, predictions_inverse)

    model_path = ARTIFACT_DIR / "lstm_model.keras"
    scaler_path = ARTIFACT_DIR / "lstm_scaler.pkl"
    model.save(model_path)
    import joblib
    joblib.dump(scaler, scaler_path)

    logger.info("Saved model to %s", model_path)
    logger.info("Saved scaler to %s", scaler_path)

    return {
        "model": model,
        "scaler": scaler,
        "history": history,
        "x_test": x_test,
        "y_test_scaled": y_test,
        "predictions_scaled": predictions.ravel(),
        "predictions": predictions_inverse,
        "actual": actual_inverse,
        "mae": mae,
        "mse": mse,
        "model_path": model_path,
        "scaler_path": scaler_path,
    }


def predict_future(data, model, scaler):
    """Predict the next value from a dict or DataFrame input containing close prices."""
    if isinstance(data, dict):
        input_frame = pd.DataFrame([data])
    elif isinstance(data, pd.DataFrame):
        input_frame = data.copy()
    else:
        raise TypeError("data must be a dict or pandas DataFrame")

    input_frame.columns = input_frame.columns.astype(str).str.strip().str.lower()
    if "close" not in input_frame.columns:
        raise ValueError("Input data must contain a 'close' column.")

    close_values = pd.to_numeric(input_frame["close"], errors="coerce").dropna().to_numpy(dtype=np.float32).reshape(-1, 1)
    scaled_close = scaler.transform(close_values)

    if len(scaled_close) < WINDOW_SIZE:
        pad_size = WINDOW_SIZE - len(scaled_close)
        scaled_close = np.vstack([np.repeat(scaled_close[:1], pad_size, axis=0), scaled_close])

    x_input = scaled_close[-WINDOW_SIZE:].reshape(1, WINDOW_SIZE, 1)
    predicted_scaled = model.predict(x_input, verbose=0)
    predicted_value = scaler.inverse_transform(predicted_scaled.reshape(-1, 1))[0, 0]
    return float(predicted_value)

In [ ]:
df = get_input_dataframe()
print("Input rows:", len(df))
print("Columns:", list(df.columns[:10]))
df.head()

In [ ]:
artifacts = train_lstm_pipeline(df)

print("MAE:", round(artifacts["mae"], 4))
print("MSE:", round(artifacts["mse"], 4))
print("Model saved to:", artifacts["model_path"])
print("Scaler saved to:", artifacts["scaler_path"])

In [ ]:
history = artifacts["history"]
predictions = artifacts["predictions"]
actual = artifacts["actual"]

plt.figure(figsize=(8, 4))
plt.plot(history.history["loss"], marker="o")
plt.title("Training Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.tight_layout()
plt.show()

preview_length = min(100, len(actual))
plt.figure(figsize=(10, 4))
plt.plot(actual[:preview_length], label="Actual")
plt.plot(predictions[:preview_length], label="Predicted")
plt.title("Actual vs Predicted Values")
plt.xlabel("Time Step")
plt.ylabel("Price")
plt.legend()
plt.tight_layout()
plt.show()

comparison = pd.DataFrame({
    "actual": actual[:10],
    "predicted": predictions[:10],
})
print("Sample Predictions:")
print(comparison.to_string(index=False))

In [ ]:
demo_input = df.tail(WINDOW_SIZE + 1).copy()
predicted_next_value = predict_future(demo_input, artifacts["model"], artifacts["scaler"])
print("Future prediction:", round(predicted_next_value, 4))

# Deployment note:
# Load ml_pipeline/models/lstm_model.keras and ml_pipeline/models/lstm_scaler.pkl for future inference.
# The predict_future function can be reused in an API endpoint or batch scoring job.

### Saved Artifacts
- `ml_pipeline/models/lstm_model.keras`
- `ml_pipeline/models/lstm_scaler.pkl`

This notebook is ready for reuse in training, batch scoring, or API deployment.